#  Mengisi Missing Value
### Mengisi Missing Value menggunakan WKNN
Weighted K-Nearest Neighbor Imputation (WKNNI), Imputasi Berbobot dengan K-Tetangga Terdekat (WKNNI) merupakan pengembangan dari metode KNNI standar untuk mengisi data hilang dengan memilih instans yang memiliki kemiripan jarak terhadap instans tidak lengkap. Berbeda dengan KNNI standar yang menggunakan rata-rata sederhana, WKNNI memberikan bobot berbeda di mana tetangga yang lebih dekat (lebih mirip) memiliki pengaruh lebih besar dibandingkan tetangga yang jauh. Ukuran kemiripan ini didefinisikan sebagai kebalikan dari jarak Euclidean, sehingga semakin kecil jaraknya, semakin besar bobot kemiripannya ($s_i$). Perhitungan jarak hanya dilakukan pada atribut yang teramati untuk memastikan validitas perbandingan antar objek.

$$
d(x,y) = \sqrt{\sum_{j \in O} (x_j - y_j)^2}
$$

$$
s_i = \frac{1}{d_i}
$$

Nilai estimasi untuk data yang hilang dihitung menggunakan rata-rata tertimbang (Weighted Average), di mana hasil perkalian bobot dengan nilai tetangga dibagi dengan total seluruh bobot kesamaan. Logikanya, tetangga yang sangat mirip akan mendominasi hasil akhir prediksi. Karena tidak ada kriteria teoretis baku untuk menentukan jumlah tetangga terbaik, nilai $K$ harus ditentukan secara empiris melalui percobaan pada dataset.

$$\hat{y} = \frac{\sum s_i y_i}{\sum s_i}$$

#### Penentuan Parameter K

Tidak ada rumus teoritis baku untuk menentukan jumlah tetangga terbaik (**K**) pada metode **K-Nearest Neighbor**. Nilai **K** biasanya ditentukan secara empiris dengan mencoba beberapa nilai seperti **3, 5, atau 7**, kemudian dipilih nilai yang memberikan tingkat kesalahan paling kecil pada dataset.

---

#### Data

**Data Asli**

| ID | IPK | PO | JML |
|----|----|----|----|
|1|2|200000|2|
|2|3|300000|3|
|3|4|200000|2|
|4|2|200000|3|
|5|3|300000|2|
|6|4|400000|3|
|7|2|300000|?|

Data Target:

IPK = **2**  
PO = **300000**  
JML = **?** (nilai yang ingin diprediksi)

---

#### Data Setelah Normalisasi Menggunakan Min-Max

| ID | IPK | PO | JML |
|----|----|----|----|
|1|0|0|0|
|2|0.5|0.5|1|
|3|1|0|0|
|4|0|0|1|
|5|0.5|0.5|0|
|6|1|1|1|
|7|0|0.5|?|

---

#### Langkah 1 : Hitung Jarak dan Kemiripan

Perhitungan jarak menggunakan **Euclidean Distance**.

$$
d(x,y) = \sqrt{(x_1-y_1)^2 + (x_2-y_2)^2}
$$

Kemiripan dihitung dengan:

$$
s_i = \frac{1}{d_i}
$$

| Tetangga | Perhitungan Jarak | Jarak $(d_i)$ | Kemiripan $(s_i=\frac{1}{d_i})$ |
|---------|------------------|------|------|
|1|$\sqrt{(0-0)^2+(0.5-0)^2}$|0.5|2|
|2|$\sqrt{(0-0.5)^2+(0.5-0.5)^2}$|0.5|2|
|3|$\sqrt{(0-1)^2+(0.5-0)^2}$|1.118|0.895|
|4|$\sqrt{(0-0)^2+(0.5-0)^2}$|0.5|2|
|5|$\sqrt{(0-0.5)^2+(0.5-0.5)^2}$|0.5|2|
|6|$\sqrt{(0-1)^2+(0.5-1)^2}$|1.118|0.895|

---

#### Langkah 2 : Hitung Estimasi Menggunakan Weighted Average

Estimasi nilai dihitung menggunakan rumus:

$$
\hat{y} = \frac{sum s_i y_i}{\sum s_i}
$$

Dimana:

- \(s_i\) = kemiripan (bobot)
- \(y_i\) = nilai target

---

**A. Pembilang $\sum s_i y_i$**

| Bobot \(s_i\) | JML \(y_i\) | Hasil \(s_i \times y_i\) |
|------|------|------|
|2|0|0|
|2|1|2|
|0.895|0|0|
|2|1|2|
|2|0|0|
|0.895|1|0.895|

Total Pembilang:

$$
0 + 2 + 0 + 2 + 0 + 0.895
$$

$$
= 4.895
$$

---

**B. Penyebut $\sum s_i$**

$$
2 + 2 + 0.895 + 2 + 2 + 0.895
$$

$$
= 9.79
$$

---

**Hasil Akhir Skala Ternormalisasi**

$$
\hat{y} =
\frac{4.895}{9.79}
$$

$$
= 0.5
$$

Sehingga nilai **JML yang hilang diprediksi dalam skla ternormalisasi yaitu 0.5**.





#### Langkah 3 : Denormalisasi (Mengembalikan ke Skala Asli)

Setelah proses prediksi menggunakan metode **WKNN**, diperoleh nilai hasil prediksi dalam bentuk **ternormalisasi**, yaitu:

$$
\hat{y} = 0.5
$$

Namun, nilai tersebut masih dalam skala normalisasi (0–1), sehingga perlu dikembalikan ke **skala asli** agar sesuai dengan data awal.

Diketahui:
- Nilai minimum (min) = 2  
- Nilai maksimum (max) = 3  

#### Rumus Denormalisasi

$$
y_{asli} = \hat{y} \times (max - min) + min
$$

#### Perhitungan

$$
y_{asli} = 0.5 \times (3 - 2) + 2
$$

$$
y_{asli} = 0.5 \times 1 + 2
$$

$$
y_{asli} = 2.5
$$

---

#### Kesimpulan

Berdasarkan perhitungan menggunakan metode **Weighted K-Nearest Neighbor (WKNN)**, nilai **JML** yang hilang pada data ke-7 diprediksi sebesar:

$$
\boxed{2.5}
$$

In [7]:
import pandas as pd
import numpy as np
from sklearn.preprocessing import MinMaxScaler
from sklearn.neighbors import KNeighborsRegressor as KNR

data = {
    'IPK': [2, 3, 4, 2, 3, 4, 2],
    'PO': [2000000, 3000000, 2000000, 2000000, 3000000, 4000000, 3000000],
    'JML': [2, 3, 2, 3, 2, 3, np.nan]
}
df = pd.DataFrame(data)

df_train = df.iloc[:6].copy()
df_test = df.iloc[6:].copy()

scaler_features = MinMaxScaler()
scaler_target = MinMaxScaler()

train_X = scaler_features.fit_transform(df_train[['IPK', 'PO']])
test_X = scaler_features.transform(df_test[['IPK', 'PO']])

print("DATA NORMALISASI")
train_X_df = pd.DataFrame(train_X, columns=["IPK", "PO"])
print(train_X_df)

train_y = scaler_target.fit_transform(df_train[['JML']])

def custom_weights(distances):
    return 1 / (distances**2)

knn = KNR(n_neighbors=6, weights=custom_weights, metric='euclidean')
knn.fit(train_X, train_y)

pred_scaled = knn.predict(test_X)
pred_final = scaler_target.inverse_transform(pred_scaled)

print(f"\nPrediksi nilai JML Ternormalisasi: {pred_scaled[0][0]}")
print(f"Hasil Prediksi JML Denormlaisasi: {pred_final[0][0]}")

DATA NORMALISASI
   IPK   PO
0  0.0  0.0
1  0.5  0.5
2  1.0  0.0
3  0.0  0.0
4  0.5  0.5
5  1.0  1.0

Prediksi nilai JML Ternormalisasi: 0.5
Hasil Prediksi JML Denormlaisasi: 2.5
